In [1]:
%load_ext autoreload

In [49]:
%autoreload 2
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import plotly.express as px
from itertools import product

from darpinstances.results import load_aggregate_stats_in_dir, load_occupancies_in_dir
from darpinstances.instance_generation.convert_formats import RESOURCE_PATH

## notes
- doesn't make sense to have dependent variable on x axis and independent variable on y axis -> it should be the other way around
- doesn't make sense to compare it across all instances (different config parameters) - choose one where all methods finish and where the instances are same except for on dependent variable that we want to compare it for
    - eg. occupancy based on delay - the instances should be same in: area, capacity, size (duration, sample) and if for all methods then the methods all need to finish
- could be configured: size (duration, sample), capacity, area, delay, method

# path setup

In [3]:
# run_id = "1-run-23-3"
run_id = "2-run-10-4"

In [4]:
PATH = Path.cwd()
current_path = PATH
INSTANCE_PATH = PATH.parents[2] / "Instances"
RESULTS_PATH = PATH.parents[2] / "final-results" / run_id / "Results"

In [5]:
PATH = PATH.parents[2]
os.chdir(PATH)
IMG_PATH = PATH / "Ridesharing_DARP_instances/figures/bc-dominika"

In [6]:
Path.cwd()

PosixPath('/home/dominika/Desktop/deathOFbachelor')

In [7]:
plt.rc('font', size=20)

In [ ]:
PATH

## results dataframe setup

In [94]:
# areas = ['Porto']
# areas = ['Sydney']
areas = ['Porto', 'Sydney', 'DC', 'Manhattan', 'Chicago', 'NYC']

### prepare occupancy dataframe

In [95]:
oc_df = pd.DataFrame()
for area in areas:
    res_in_area = load_occupancies_in_dir(RESULTS_PATH / area)
    if res_in_area is None:
        continue
    res_in_area['area'] = area
    oc_df = pd.concat([oc_df, res_in_area], ignore_index=True)
    # os.chdir(PATH)

16:30:38 [INFO] Loading occupancy stats in /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto
16:30:38 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-solution.json
16:30:38 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-performance.json
16:30:38 [INFO] Loading experiment config from /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml
16:30:38 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_6/config.yaml
16:30:38 [INFO] Loading json file from: /home/dominika/Desktop/dea

In [96]:
oc_df['cost_per_request'] = oc_df['cost_minutes'] * 60 / oc_df['req_count']
oc_df['area_short'] = oc_df['area'].map({
    'Porto': 'PT',
    'Sydney': 'SY',
    'DC': 'DC',
    'Manhattan': 'MH',
    'Chicago': 'CH',
    'NYC': 'NY'
})
area_order = {
    'Porto': 0,
    'Sydney': 1,
    'DC': 2,
    'Manhattan': 3,
    'Chicago': 4,
    'NYC': 5
}
oc_df['area_order'] = oc_df['area'].map(area_order)
oc_df.sort_values(by=['area_order', 'duration_minutes', 'max_delay', 'method'], inplace=True)

In [89]:
oc_df

,cost_minutes,total_time,dropped_requests,solver_stats,avg_delay,plan_count,req_count,avg_occupancy,used_connections,total_driving_duration,...,start_time,end_time,capacity,duration_minutes,occupancy,vehicle_hours,area,cost_per_request,area_short,area_order
253,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,0,3.813889,DC,1140.000000,DC,2
254,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,1,12.735278,DC,1140.000000,DC,2
255,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,2,0.551389,DC,1140.000000,DC,2
256,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,3,0.000000,DC,1140.000000,DC,2
257,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,4,0.000000,DC,1140.000000,DC,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1513,13241,726.163,0,"{'VGA_time': 879, 'chaining_time': 725268, 'gr...",1200.388753,350,1227,1.848010,1903,794479,...,2022-04-05 18:00:00,2022-04-05 20:00:00,4,120,6,0.000000,DC,647.481663,DC,2
1514,13241,726.163,0,"{'VGA_time': 879, 'chaining_time': 725268, 'gr...",1200.388753,350,1227,1.848010,1903,794479,...,2022-04-05 18:00:00,2022-04-05 20:00:00,4,120,7,0.000000,DC,647.481663,DC,2
1515,13241,726.163,0,"{'VGA_time': 879, 'chaining_time': 725268, 'gr...",1200.388753,350,1227,1.848010,1903,794479,...,2022-04-05 18:00:00,2022-04-05 20:00:00,4,120,8,0.000000,DC,647.481663,DC,2
1516,13241,726.163,0,"{'VGA_time': 879, 'chaining_time': 725268, 'gr...",1200.388753,350,1227,1.848010,1903,794479,...,2022-04-05 18:00:00,2022-04-05 20:00:00,4,120,9,0.000000,DC,647.481663,DC,2


### prepare basic dataframe

In [48]:
df = pd.DataFrame()
for area in areas:
    res_in_area = load_aggregate_stats_in_dir(RESULTS_PATH / area)
    if res_in_area is None:
        continue
    res_in_area['area'] = area
    df = pd.concat([df, res_in_area], ignore_index=True)
    # os.chdir(PATH)

09:49:54 [INFO] Loading aggregate stats in /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto


09:49:54 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-solution.json
09:49:54 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-performance.json
09:49:54 [INFO] Loading experiment config from /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml
09:49:54 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_6/config.yaml
09:49:54 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/halns/config.

In [50]:
df['cost_per_request'] = df['cost_minutes'] * 60 / df['req_count']
df['area_short'] = df['area'].map({
    'Porto': 'PT',
    'Sydney': 'SY',
    'DC': 'DC',
    'Manhattan': 'MH',
    'Chicago': 'CH',
    'NYC': 'NY'
})
area_order = {
    'Porto': 0,
    'Sydney': 1,
    'DC': 2,
    'Manhattan': 3,
    'Chicago': 4,
    'NYC': 5
}
df['area_order'] = df['area'].map(area_order)
df.sort_values(by=['area_order', 'duration_minutes', 'max_delay', 'method'], inplace=True)

In [51]:
df

,method,cost_minutes,total_time,avg_delay,dropped_requests,req_count,plan_count,total_driving_duration,total_waiting_duration,avg_waiting_duration,...,duration_minutes,max_delay,start_time,end_time,trip_durations,capacity,area,cost_per_request,area_short,area_order
25,halns,120,4.760,296.894737,0,19,10,7220,130,13.000000,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[288, 458, 189, 129, 127, 572, 247, 276, 276, ...",6,Porto,378.947368,PT,0
29,halns,120,4.858,296.894737,0,19,10,7220,130,13.000000,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[288, 458, 189, 129, 127, 572, 247, 276, 276, ...",10,Porto,378.947368,PT,0
33,halns,120,4.877,296.894737,0,19,10,7220,130,13.000000,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[288, 458, 189, 129, 127, 572, 247, 276, 276, ...",4,Porto,378.947368,PT,0
26,ih,142,0.000,295.631579,0,19,11,8502,4,0.363636,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[653, 129, 127, 572, 273, 470, 122, 746, 99, 1...",6,Porto,448.421053,PT,0
30,ih,142,0.000,295.631579,0,19,11,8502,4,0.363636,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[653, 129, 127, 572, 273, 470, 122, 746, 99, 1...",10,Porto,448.421053,PT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578,ih,449820,3217.594,806.070835,0,75796,9427,26989208,2050107,217.471836,...,120,600,2022-04-05 18:00:00,2022-04-05 20:00:00,"[373, 524, 1832, 2385, 1895, 1000, 2362, 201, ...",10,NYC,356.076838,NY,5
579,ih,477049,2991.034,789.923809,0,75796,9676,28622930,2064051,213.316556,...,120,600,2022-04-05 18:00:00,2022-04-05 20:00:00,"[699, 433, 792, 246, 1569, 1431, 294, 725, 230...",4,NYC,377.631273,NY,5
580,ih,418319,3077.935,884.301757,0,75796,7398,25099116,1192176,161.148418,...,120,900,2022-04-05 18:00:00,2022-04-05 20:00:00,"[698, 331, 765, 2092, 323, 253, 469, 968, 553,...",6,NYC,331.140693,NY,5
581,ih,404334,2783.139,898.169653,0,75796,7251,24260060,1259428,173.690250,...,120,900,2022-04-05 18:00:00,2022-04-05 20:00:00,"[522, 804, 600, 870, 860, 484, 308, 1176, 884,...",10,NYC,320.070188,NY,5


### average cost per request (old)

In [ ]:
dfih = df[df['method'] == 'ih']
fig = px.bar(
    dfih,
    x = 'area_short',
    y = 'cost_per_request',
    barmode='group',
    title = 'Average Cost per Request',
    facet_col='duration_minutes',
    facet_row='max_delay'
)

# shared axes titles
fig.for_each_yaxis(lambda y: y.update(title = ''))
fig.add_annotation(x=-0.06, y=0.5, text="travel time per request [s]", textangle=-90, xref="paper", yref="paper", showarrow=False)
fig.for_each_xaxis(lambda y: y.update(title = ''))

# faceting annotations
fig.add_annotation(x=0.5, y=1.15, text="instance length [min]",  xref="paper", yref="paper", showarrow=False)
fig.add_annotation(x=1.02, y=0.5, text="maximum delay [s]",  xref="paper", yref="paper", showarrow=False, textangle=90)

# faceting label editing
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [ ]:
dfih['cost_per_request'].describe()

# Basic info

In [ ]:
df['trip_durations'] = df['trip_durations'].apply(lambda x: np.array(x))
all_durations = np.concatenate(df['trip_durations'].values)

all_durations_minutes = all_durations / 60

sorted_durations = np.sort(all_durations_minutes)

In [ ]:
df['trip_durations_mins'] = df['trip_durations'].apply(lambda x: (np.array(x) / 60).round())
all_durations_mins = np.concatenate(df['trip_durations_mins'].values)
trip_count = len(all_durations_mins)

In [ ]:
bin_edges = list(range(0, 31, 5)) + [float('inf')]

In [ ]:
counts, edges = np.histogram(all_durations_mins, bins=bin_edges)
percentages = counts / len(all_durations_mins) * 100

In [ ]:
plt.figure()
# plt.figure(figsize=(5, 3))
plt.bar(range(len(counts)), percentages, align='edge', width=1, edgecolor='black')
plt.xlabel("Trip duration [min]")
plt.ylabel("% of total trips")
# plt.title("Trip Duration Distribution")

tick_positions = np.arange(len(edges) - 1)  # One tick for each bin edge
tick_labels = [f"{int(edges[i])}" for i in range(len(edges) - 2)] + ["30+"]
plt.xticks(tick_positions, tick_labels)

plt.savefig(f'{area}_trip_duration_histogram.png', bbox_inches='tight')

plt.show()

In [ ]:
df['total_driving_duration_minutes'] = df['total_driving_duration'] / 60
df['driving_duration_mins_per_vehicle'] = df['total_driving_duration_minutes']/df['plan_count']


In [ ]:
# Create a scatter plot
# plt.figure(figsize=(10, 6))

x_values = (df['avg_delay']/60).values
y1_values = df['avg_occupancy'].values

# Sort data by avg_delay to align the plots properly
sorted_indices = x_values.argsort()
x_values = x_values[sorted_indices]
y1_values = y1_values[sorted_indices]

plt.figure()
plt.plot(x_values, y1_values, color='orange', linewidth=2)
plt.xlabel('Average Delay [min]')
plt.ylabel('Average Occupancy')
# plt.grid(True)
plt.savefig(f'{area}_delay_occupancy.png', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Prepare data
x_values = (df['avg_delay']/60).values
y1_values = df['avg_occupancy'].values
y2_values = df['driving_duration_mins_per_vehicle'].values

# Sort data by avg_delay to align the plots properly
sorted_indices = x_values.argsort()
x_values = x_values[sorted_indices]
y1_values = y1_values[sorted_indices]
y2_values = y2_values[sorted_indices]

# Create the figure and axis
fig, ax1 = plt.subplots(figsize=(12, 6))

bar_positions = np.arange(len(x_values))

# Plot bar graph for driving duration
ax1.bar(bar_positions, y2_values, color='skyblue', alpha=0.7, label='Driving Duration (mins)', width=0.8)
ax1.set_ylabel('Driving Duration (mins)', color='skyblue')
ax1.set_xlabel('Average Delay (seconds)')
ax1.tick_params(axis='y', labelcolor='skyblue')

x_ticks = np.round(x_values, 1)  # Round avg_delay to 1 decimal place for clarity
ax1.set_xticks(bar_positions)
ax1.set_xticklabels([f"{tick:.1f}" for tick in x_ticks])

# Create a second y-axis for the line plot
ax2 = ax1.twinx()
ax2.plot(bar_positions, y1_values, color='orange', marker='o', linewidth=2, label='Average Occupancy')
ax2.set_ylabel('Average Occupancy', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

# Title and grid
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Add legends
# fig.legend(loc='upper left', bbox_to_anchor=(0, 1), bbox_transform=ax1.transAxes)

# Show the plot
# plt.savefig(f'{area}_trip_dur_occupancy_delay.png', bbox_inches='tight')
plt.tight_layout()
plt.show()


In [ ]:
oc_df

In [ ]:
driving_hours = (oc_df['total_driving_duration']/60/60).round(2)
vehicle_hours = oc_df['vehicle_hours']
occupancies = oc_df['occupancy']
vehicle_hours_percentage = (vehicle_hours / driving_hours) * 100
oc_df['vehicle_hours_percentage'] = vehicle_hours_percentage
oc_df['driving_hours'] = driving_hours

In [ ]:
grouped = oc_df.pivot(index='driving_hours', columns='occupancy', values='vehicle_hours_percentage')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

bar_width = 0.15
positions = np.arange(len(grouped))
for i in range(5):
    ax.bar(positions + i * bar_width, grouped[i], width=bar_width, label=f'{i}')

ax.set_xlabel('Driving durations (hours)')
ax.set_ylabel('% of total trip durations')

ax.set_xticks(positions + 2 * bar_width)
ax.set_xticklabels(grouped.index, rotation=45)
ax.legend(title='Occupancy level', bbox_to_anchor=(1, 1))

plt.savefig(f'{area}_occupancy_driving_duration.png', bbox_inches='tight')
# plt.show()

# Demand statistics

## Porto

In [ ]:
from darpinstances.instance_generation.demand_to_database import load_data_from_csv
from darpinstances.instance_generation.convert_formats import RESOURCE_PATH
city = 'Porto'
csvfile = RESOURCE_PATH / f'{city}_trips.csv'
df = load_data_from_csv(csvfile)

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['Year/Month'] = df['timestamp'].dt.to_period('M')
df['Day'] = df['timestamp'].dt.date

In [ ]:
trips_per_month = df.groupby('Year/Month').size().reset_index(name='Trips Per Month')
trips_per_day = df.groupby(['Year/Month', 'Day']).size().reset_index(name='Trips Per Day')
days_per_month = trips_per_day.groupby('Year/Month').size().reset_index(name='Days Per Month')
merged_trips = pd.merge(trips_per_month, days_per_month, on='Year/Month')
merged_trips['Avg Trips Per Day'] = merged_trips['Trips Per Month'] / merged_trips['Days Per Month']

In [ ]:
new_columns = ['Year/Month', 'Trips Per Month', 'Avg Trips Per Day']
stat_df = pd.DataFrame(columns=new_columns)
stat_df['Year/Month'] = merged_trips['Year/Month'].astype(str)
stat_df['Trips Per Month'] = merged_trips['Trips Per Month']
stat_df['Avg Trips Per Day'] = merged_trips['Avg Trips Per Day'].round()

In [ ]:
year_month_np = stat_df['Year/Month'].to_numpy()
avg_trips_per_day_np = stat_df['Avg Trips Per Day'].to_numpy()
trips_per_month_np = stat_df['Trips Per Month'].to_numpy()

In [ ]:
# Plot the average trips per day
plt.figure(figsize=(12, 6))
plt.plot(year_month_np, avg_trips_per_day_np)
plt.xlabel('Year/Month')
plt.ylabel('Avg Trips Per Day')
# plt.title('Average Trips Per Day Over Time')
plt.xticks(rotation=45)
plt.savefig(f'{city}_avg_trips_per_day.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(year_month_np, trips_per_month_np)
plt.xlabel('Year/Month')
plt.ylabel('Trips Per Month')
# plt.title('Trips Per Month Over Time')
plt.xticks(rotation=45)
plt.savefig(f'{city}_trips_per_month.png', bbox_inches='tight')
plt.show()

### Daily stats

In [ ]:
start_date = pd.to_datetime('2013-07-01')
end_date = pd.to_datetime('2013-07-07')
df_week = df[(df['timestamp'] >= start_date) & (df['timestamp'] <= end_date)]

In [ ]:
df_week.loc[:, 'Date'] = df_week['timestamp'].dt.date
df_week.loc[:, 'Hour'] = df_week['timestamp'].dt.hour

trips_per_hour = df_week.groupby(['Date', 'Hour']).size().reset_index(name='Trips Per Hour')

trips_per_hour['Datetime'] = pd.to_datetime(trips_per_hour['Date'].astype(str) + ' ' + trips_per_hour['Hour'].astype(str) + ':00:00')

datetimes = trips_per_hour['Datetime'].values
trips_per_hour_values = trips_per_hour['Trips Per Hour'].values

In [ ]:
plt.figure(figsize=(30, 8))
plt.plot(datetimes, trips_per_hour_values, linewidth=3)
plt.xlabel('Date and Hour')
plt.ylabel('Number of Trips')
# plt.title('Number of Trips Per Hour from 2013-07-01 to 2013-07-07')
plt.xticks(rotation=45)
plt.savefig(f'{city}_trips_per_hour.png', bbox_inches='tight')
plt.show()

### 2013-07-01

In [ ]:
df_oneday = df[df['timestamp'].dt.date == pd.to_datetime('2013-07-01').date()]

### hourly stats

In [ ]:
df_oneday.loc[:, 'Hour'] = df_oneday['timestamp'].dt.hour
trips_per_hour = df_oneday.groupby('Hour').size().reset_index(name='Trips Per Hour')
hours = trips_per_hour['Hour'].to_numpy()
trips_per_hour_values = trips_per_hour['Trips Per Hour'].to_numpy()

In [ ]:
plt.plot(hours, trips_per_hour_values)
plt.xlabel('Hour of the Day')
plt.ylabel('Number of Trips')
# plt.title('Number of Trips Per Hour')
plt.savefig(f'{city}_trips_per_hour.png', bbox_inches='tight')
plt.show()

## Sydney

In [ ]:
city = 'Sydney'

In [ ]:
csvfile = RESOURCE_PATH / f'{city}_trips.csv'
df = load_data_from_csv(csvfile)

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['Year/Month'] = df['timestamp'].dt.to_period('M')
df['Day'] = df['timestamp'].dt.date

### 2014-01-01

In [ ]:
df_oneday = df[df['timestamp'].dt.date == pd.to_datetime('2014-01-01').date()]

### hourly stats

In [ ]:
df_oneday.loc[:, 'Hour'] = df_oneday['timestamp'].dt.hour
trips_per_hour = df_oneday.groupby('Hour').size().reset_index(name='Trips Per Hour')
hours = trips_per_hour['Hour'].to_numpy()
trips_per_hour_values = trips_per_hour['Trips Per Hour'].to_numpy()

In [ ]:
plt.plot(hours, trips_per_hour_values)
plt.xlabel('Hour of the Day')
plt.ylabel('Number of Trips')
plt.savefig(f'{city}_trips_per_hour.png', bbox_inches='tight')
plt.show()

# Methods performance

## Missing results of methods for areas

In [45]:
df

,method,cost_minutes,total_time,avg_delay,dropped_requests,req_count,plan_count,total_driving_duration,total_waiting_duration,avg_waiting_duration,...,duration_minutes,max_delay,start_time,end_time,trip_durations,capacity,area,cost_per_request,area_short,area_order
23,halns,1026,17.022,922.814815,0,54,47,61562,15,0.319149,...,5,180,2022-04-05 18:00:00,2022-04-05 18:05:00,"[2121, 772, 1826, 752, 213, 403, 648, 334, 163...",6,DC,1140.000000,DC,2
27,halns,1001,15.938,922.814815,0,54,47,60040,15,0.319149,...,5,180,2022-04-05 18:00:00,2022-04-05 18:05:00,"[2121, 772, 1826, 752, 213, 403, 648, 334, 163...",10,DC,1112.222222,DC,2
31,halns,999,16.390,922.814815,0,54,47,59946,15,0.319149,...,5,180,2022-04-05 18:00:00,2022-04-05 18:05:00,"[2121, 772, 1826, 752, 213, 403, 648, 334, 163...",4,DC,1110.000000,DC,2
24,ih,1033,0.001,924.777778,0,54,47,61958,0,0.000000,...,5,180,2022-04-05 18:00:00,2022-04-05 18:05:00,"[1792, 217, 1747, 1051, 329, 404, 648, 215, 17...",6,DC,1147.777778,DC,2
28,ih,1013,0.001,924.777778,0,54,47,60774,0,0.000000,...,5,180,2022-04-05 18:00:00,2022-04-05 18:05:00,"[1136, 203, 403, 1519, 1887, 1144, 1382, 1086,...",10,DC,1125.555556,DC,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,ih,12524,0.639,1298.603912,0,1227,238,751436,77259,324.617647,...,120,900,2022-04-05 18:00:00,2022-04-05 20:00:00,"[105, 604, 633, 1522, 1112, 371, 1702, 2076, 1...",10,DC,612.420538,DC,2
138,ih,12631,0.620,1307.652812,0,1227,245,757859,94729,386.648980,...,120,900,2022-04-05 18:00:00,2022-04-05 20:00:00,"[1232, 2567, 2146, 963, 497, 1130, 1079, 588, ...",4,DC,617.652812,DC,2
133,vga_chaining,13241,651.989,1203.614507,0,1227,342,794432,182086,532.415205,...,120,900,2022-04-05 18:00:00,2022-04-05 20:00:00,"[1231, 2374, 652, 1676, 2115, 2565, 120, 1144,...",6,DC,647.481663,DC,2
135,vga_chaining,13243,810.152,1203.858191,0,1227,336,794570,183109,544.967262,...,120,900,2022-04-05 18:00:00,2022-04-05 20:00:00,"[1231, 2374, 652, 1676, 2115, 2565, 120, 1144,...",10,DC,647.579462,DC,2


In [52]:
df_filtered = df[df['cost_per_request'] > 0][['max_delay', 'method', 'area_short', 'cost_per_request', 'duration_minutes', 'capacity']].drop_duplicates()

In [53]:
# Generate all combinations of method, area_short, duration_minutes, and max_delay
methods = df_filtered['method'].unique()
areas = df_filtered['area_short'].unique()
durations = df_filtered['duration_minutes'].unique()
delays = df_filtered['max_delay'].unique()

all_combinations = pd.DataFrame(
    list(product(methods, areas, durations, delays)),
    columns=['method', 'area_short', 'duration_minutes', 'max_delay']
)

# Merge with the filtered data to find missing combinations
df_complete = all_combinations.merge(df_filtered, on=['method', 'area_short', 'duration_minutes', 'max_delay'], how='left')

# Add a column to mark missing values
df_complete['is_missing'] = df_complete['cost_per_request'].isna()

# Replace missing cost_per_request with 0 for plotting
df_complete['cost_per_request'] = df_complete['cost_per_request'].fillna(0)


In [54]:
method_order = sorted(df_complete['method'].unique())
method_offsets = {
    method: i - (len(method_order) - 1) / 2  # center around 0
    for i, method in enumerate(method_order)
}

vals_to_axis = {}
i = 1
delays = sorted(df_complete['max_delay'].unique(), reverse=True)
durations = sorted(df_complete['duration_minutes'].unique())
for max_delay in delays:
    for duration in durations:
        vals_to_axis[(duration, max_delay)] = i
        i += 1

In [55]:
capacity_df = df_complete[df_complete['capacity'] == 4]

In [ ]:
fig = px.bar(
    df_complete,
    # capacity_df,
    x='area_short',
    y='cost_per_request',
    color='method',
    barmode='group',
    title='Average cost per request by method and area',
    facet_col='duration_minutes',
    facet_row='max_delay',
    # pattern_shape='capacity',
    # pattern_shape_sequence=['.', 'x', '+']
    # text='capacity'
)

# Drop duplicates for spacing reference
unique_areas = df_complete['area_short'].unique()

# Add X annotations for missing method/area combos
for _, row in df_complete[df_complete['is_missing']].iterrows():
    axis_ref = vals_to_axis[(row['duration_minutes'], row['max_delay'])]

    xref = f'x{axis_ref}' if axis_ref > 1 else 'x'
    yref = f'y{axis_ref}' if axis_ref > 1 else 'y'

    offset = method_offsets[row['method']] * 0.3  # tweak this for spacing
    fig.add_annotation(
        x=row['area_short'],
        y=0,
        text="X",
        xanchor='center',
        yanchor='bottom',
        showarrow=False,
        font=dict(color='black', size=10),
        xref=xref,
        yref=yref,
        xshift=offset * 50  # pixel offset for visual spacing
    )

# Shared axes titles
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.04, y=0.5, text="Average cost per request [s]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.5, y=1.15, text="Instance length [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.02, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.update_traces(textfont_size=7)

fig.show()

### TODO: do the same for subsampling data

## Delay

each area has different mean for average delay - does it have anything to do with the infrastructure? what's the highway/area ratio?

In [406]:
delay_area = oc_df[['area_short', 'max_delay', 'avg_delay']]
fig = px.histogram(
    delay_area,
    x='avg_delay',
    facet_col='area_short',
    facet_row='max_delay',
    title='Average delay by area and maximum delay')

fig.update_layout(
    height=800,
    width=1400,
)

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
fig.add_annotation(x=0.5, y=1.05, text="Area", xref="paper", yref="paper", showarrow=False) # facet column title
fig.add_annotation(x=1.02, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.04, y=0.5, text="Count", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.08, text="Average delay [s]", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [421]:
delay_area = oc_df[['area_short', 'max_delay', 'avg_delay']]
fig = px.histogram(
    delay_area,
    x='avg_delay',
    color='area_short',
    title='Average delay by area and maximum delay',
    marginal='box'
    )

# fig.update_layout(
#     height=800,
#     width=1400,
# )

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
# fig.add_annotation(x=0.5, y=1.15, text="Area", xref="paper", yref="paper", showarrow=False) # facet column title
# fig.add_annotation(x=1.02, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.03, y=0.5, text="Count", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.15, text="Average delay [s]", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## Occupancy

In [223]:
oc_method = oc_df[['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity', 'vehicle_hours', 'occupancy']].drop_duplicates()

In [224]:
oc_method['weighted_occ'] = oc_method['occupancy'] * oc_method['vehicle_hours']
weighted_avg = (
    oc_method
    .groupby(['max_delay', 'method', 'area_short', 'duration_minutes'])
    .agg(
        total_weighted_occupancy=('weighted_occ', 'sum'),
        total_vehicle_hours=('vehicle_hours', 'sum')
    )
    .reset_index()
)
weighted_avg['occ_ratio'] = (
    weighted_avg['total_weighted_occupancy'] / weighted_avg['total_vehicle_hours']
)

In [216]:
weighted_avg

,max_delay,method,area_order,duration_minutes,total_weighted_occupancy,total_vehicle_hours
0,180,halns,0,5,4.592500,6.016667
1,180,halns,0,15,13.354167,15.174167
2,180,halns,0,30,26.713333,28.225833
3,180,halns,0,120,103.956667,111.305833
4,180,halns,2,5,41.514167,50.430000
...,...,...,...,...,...,...
196,900,vga_chaining,0,120,131.301667,93.076667
197,900,vga_chaining,2,5,52.208333,31.177222
198,900,vga_chaining,2,15,162.160000,91.719444
199,900,vga_chaining,2,30,325.742778,179.535000


In [190]:
oc_method_group = (
    oc_method
    .groupby(['max_delay','method', 'area_order', 'duration_minutes', 'capacity'])
    .agg(
        ('vehicle_hours', lambda x: (oc_method.loc[x.index, 'occupancy'] * x).sum()),
       {'vehicle_hours': 'sum'})
    .reset_index()
)

oc_method_group['weighted_avg_occupancy'] = oc_method_group['total_weighted_occupancy'] / oc_method_group['total_vehicle_hours']
oc_method_group = oc_method_group.reset_index()

AttributeError: 'SeriesGroupBy' object has no attribute 'vehicle_hours'

In [188]:
oc_method_group

,index,max_delay,method,area_order,duration_minutes,capacity,total_weighted_occupancy,total_vehicle_hours,weighted_avg_occupancy
0,0,180,halns,0,5,4,1.530833,2.005556,0.763296
1,1,180,halns,0,5,6,1.530833,2.005556,0.763296
2,2,180,halns,0,5,10,1.530833,2.005556,0.763296
3,3,180,halns,0,15,4,4.451389,5.058056,0.880059
4,4,180,halns,0,15,6,4.451389,5.058056,0.880059
...,...,...,...,...,...,...,...,...,...
584,584,900,vga_chaining,2,30,6,108.751944,59.889444,1.815878
585,585,900,vga_chaining,2,30,10,108.751944,59.592778,1.824918
586,586,900,vga_chaining,2,120,4,407.834722,220.688611,1.848010
587,587,900,vga_chaining,2,120,6,408.946111,220.675556,1.853155


In [ ]:
# oc_method_group = (
#     oc_method
#     .groupby(['method', 'area_order', 'duration_minutes', 'max_delay'])
#     # .groupby(['method', 'avg_occupancy', 'area_order', 'duration_minutes', 'max_delay', 'capacity'])
#     .agg({'vehicle_hours': 'sum'})
#     .reset_index()
# )
oc_method_group = (
    oc_method
    .groupby(['max_delay', 'method', 'area_order', 'duration_minutes', 'capacity'])
    .apply(lambda g: (g['occupancy'] * g['vehicle_hours']).sum() / g['vehicle_hours'].sum())
    .reset_index(name='weighted_avg_occupancy')
)
oc_method_group = oc_method

/tmp/ipykernel_11176/707918930.py:4: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [183]:
occ_ration = oc_method_group['occupancy'] / oc_method_group['vehicle_hours']
oc_method_group['occ_ratio'] = occ_ration

In [174]:
delay1 = oc_method_group[oc_method_group['max_delay'] == delays[-4]]

In [170]:
oc_method_group[oc_method_group['area_order'] == 0]

,method,avg_occupancy,area_order,duration_minutes,max_delay,capacity,vehicle_hours,occ_ratio
0,halns,0.763296,0,5,180,4,2.005556,0.380591
1,halns,0.763296,0,5,180,6,2.005556,0.380591
2,halns,0.763296,0,5,180,10,2.005556,0.380591
6,halns,0.880059,0,15,180,4,5.058056,0.173992
7,halns,0.880059,0,15,180,6,5.058056,0.173992
...,...,...,...,...,...,...,...,...
553,vga_chaining,1.527330,0,30,900,6,7.480556,0.204173
554,vga_chaining,1.527330,0,30,900,10,7.480556,0.204173
556,vga_chaining,1.544903,0,5,900,4,1.373333,1.124929
557,vga_chaining,1.544903,0,5,900,6,1.373333,1.124929


In [225]:
fig = px.bar(
    # oc_method,
    weighted_avg,
    # weighted_avg[weighted_avg['area_order'] == 1],
    # oc_method_group[oc_method_group['area_order'] == 0],
    # x='vehicle_hours',
    # y='avg_occupancy',
    y='occ_ratio',
    x='max_delay',
    barmode='group',
    color='method',
    # title='Average Cost per Request by Area and Method',
    # facet_row='area_short',
    facet_col='duration_minutes',
    facet_row='area_short'
)
fig.update_layout(
    height=1200,
    width=1800,
    title_x=0.5
)

fig.show()

In [230]:
fig = px.bar(
    # oc_method,
    weighted_avg,
    # weighted_avg[weighted_avg['area_order'] == 1],
    # oc_method_group[oc_method_group['area_order'] == 0],
    # x='vehicle_hours',
    # y='avg_occupancy',
    y='occ_ratio',
    x='area_short',
    barmode='group',
    color='method',
    # title='Average Cost per Request by Area and Method',
    facet_col='duration_minutes',
    facet_row='max_delay'
)
# fig.update_layout(
#     height=1200,
#     width=1800,
#     title_x=0.5
# )

# Shared axes titles
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.02, y=0.5, text="Occupancy ratio", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.5, y=1.15, text="Instance length [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.02, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [99]:
avg_oc_df = oc_df.copy()
avg_oc_df['avg_delay_min'] = avg_oc_df['avg_delay'] / 60
avg_oc_df = avg_oc_df.sort_values(by=['area_short', 'method', 'avg_delay_min'])

In [260]:
delays

[900, 600, 300, 180]

### occupancy for area

In [276]:
# Porto now
area_nick = 'PT'
capacity = 4
duration = 5
# delay_s = 180

In [277]:
area_avg_oc_df = avg_oc_df[avg_oc_df['area_short'] == area_nick]
area_avg_oc_df = avg_oc_df[avg_oc_df['capacity'] == capacity]
area_avg_oc_df = avg_oc_df[avg_oc_df['duration_minutes'] == duration]
# area_avg_oc_df = avg_oc_df[avg_oc_df['max_delay'] == delay_s]
sorted_occ_delay_df = area_avg_oc_df.sort_values(by=['avg_delay_min'])

#### avg occupancy based on avg delay

In [251]:
sorted_occ_delay_df = avg_oc_df.sort_values(by=['avg_delay_min'])

In [292]:
# area_nick = 'PT'
# capacity = 4
# duration = 5

# area_avg_oc_df = avg_oc_df[avg_oc_df['area_short'] == area_nick]
# area_avg_oc_df = avg_oc_df[avg_oc_df['capacity'] == capacity]
# area_avg_oc_df = avg_oc_df[avg_oc_df['duration_minutes'] == duration]
sorted_occ_delay_df = avg_oc_df.sort_values(by=['avg_delay_min'])

fig = px.line(
    sorted_occ_delay_df,
    x='avg_delay_min',
    y='avg_occupancy',
    color='method',
    line_group='method',
    title=f'Average occupancy of each method based on average delay',
    labels={
        'avg_delay_min': 'Average Delay [min]',
        'avg_occupancy': 'Average Occupancy'
    },
    facet_row='duration_minutes',
    # facet_col='duration_minutes',
    # facet_row='area_short'
    facet_col='area_short'
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))  # Clean facet labels
# fig.write_image(f"{IMG_PATH}/{area_code}_avg_occ_avg_del.png")
fig.show()

#### occupancy based driving duration

In [120]:
area_avg_oc_df['total_driving_duration_minutes'] = area_avg_oc_df['total_driving_duration'] / 60
area_avg_oc_df['driving_duration_mins_per_vehicle'] = area_avg_oc_df['total_driving_duration_minutes']/area_avg_oc_df['plan_count']

/tmp/ipykernel_11176/3280263572.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_11176/3280263572.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [121]:
area_avg_oc_df

,cost_minutes,total_time,dropped_requests,solver_stats,avg_delay,plan_count,req_count,avg_occupancy,used_connections,total_driving_duration,...,duration_minutes,occupancy,vehicle_hours,area,cost_per_request,area_short,area_order,avg_delay_min,total_driving_duration_minutes,driving_duration_mins_per_vehicle
5555,4551,0.046,0,Solver has no performance stats,844.244526,132,274,0.843318,-1,273037,...,15,0,22.180000,Chicago,996.569343,CH,4,14.070742,4550.616667,34.474369
5556,4551,0.046,0,Solver has no performance stats,844.244526,132,274,0.843318,-1,273037,...,15,1,44.140000,Chicago,996.569343,CH,4,14.070742,4550.616667,34.474369
5557,4551,0.046,0,Solver has no performance stats,844.244526,132,274,0.843318,-1,273037,...,15,2,8.758056,Chicago,996.569343,CH,4,14.070742,4550.616667,34.474369
5558,4551,0.046,0,Solver has no performance stats,844.244526,132,274,0.843318,-1,273037,...,15,3,0.758056,Chicago,996.569343,CH,4,14.070742,4550.616667,34.474369
5559,4551,0.046,0,Solver has no performance stats,844.244526,132,274,0.843318,-1,273037,...,15,4,0.007500,Chicago,996.569343,CH,4,14.070742,4550.616667,34.474369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019,10724,6.171,0,"{'VGA_time': 4475, 'chaining_time': 1695, 'gro...",787.130996,454,1084,1.308089,1383,643437,...,5,6,0.016389,Sydney,593.579336,SY,1,13.118850,10723.950000,23.621035
2020,10724,6.171,0,"{'VGA_time': 4475, 'chaining_time': 1695, 'gro...",787.130996,454,1084,1.308089,1383,643437,...,5,7,0.000000,Sydney,593.579336,SY,1,13.118850,10723.950000,23.621035
2021,10724,6.171,0,"{'VGA_time': 4475, 'chaining_time': 1695, 'gro...",787.130996,454,1084,1.308089,1383,643437,...,5,8,0.000000,Sydney,593.579336,SY,1,13.118850,10723.950000,23.621035
2022,10724,6.171,0,"{'VGA_time': 4475, 'chaining_time': 1695, 'gro...",787.130996,454,1084,1.308089,1383,643437,...,5,9,0.000000,Sydney,593.579336,SY,1,13.118850,10723.950000,23.621035


# TODO: don't compare across all instances

In [ ]:
fig = px.histogram(
    area_avg_oc_df,
    x='occupancy',
    y='vehicle_hours',
    color='method',
    barmode='group',
    title=f'{area_code}: Occupancy of each method based on driving duration',
    labels={
        'occupancy': 'Occupancy',
        'vehicle_hours': 'Driving duration [h]'
    }
)
fig.show()

### speed of methods based on capacity (TODO)

In [ ]:
fig = px.line(
    avg_oc_df[avg_oc_df['area_order'] ==0],
    x='occupancy',
    y='total_time',
    color='method',
    # barmode='group',
    title=f'Occupancy of each method based on driving duration',
    # labels={
    #     'occupancy': 'Occupancy',
    #     'vehicle_hours': 'Driving duration [h]',
    #     'area_short': 'Area'
    # },
    # facet_row='area_short',
    # facet_col='duration_minutes'
)

fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.06, y=0.5, text="Mean computational time [min]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.5, y=1.15, text="Instance size [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.02, y=0.5, text="Areas", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

### speed of methods based on vehicle count, requests count

In [122]:
df_filtered = oc_df[oc_df['cost_per_request'] > 0]

In [123]:
sort_df = df_filtered.sort_values(by=['method', 'req_count'])
# sort_df = df_filtered.sort_values(by=['method', 'total_time'])
sort_df['comp_time_min'] = sort_df['total_time'] / 60

In [124]:
fig = px.line(
    sort_df,
    # x='comp_time_min',
    # y='req_count',
    y='comp_time_min',
    x='req_count',
    color='method',
    title='Computational time vs. number of requests',
    # text="req_count",
    facet_col='area_short',
    # facet_row='area_short',
)

area_order = list(fig.layout.annotations[i].text.split('=')[-1] for i in range(len(fig.layout.annotations)))

for i, area_name in enumerate(area_order, start=1):
    req_data = sort_df[sort_df["area_short"] == area_name]  # Filter data for the current category
    bumper = 10
    if area_name == 'SY':
        bumper = 2000
    x_min = req_data["req_count"].min() - bumper
    x_max = req_data["req_count"].max() + bumper
    # x_min = req_data["comp_time_min"].min() -1
    # x_max = req_data["comp_time_min"].max() +1
    fig.update_xaxes(range=[x_min, x_max], col=i, matches=None, row=1)
    # fig.update_yaxes(range=[x_min, x_max], row=i, col=1, matches=None)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [ ]:
# sort_plan_df = df_filtered.sort_values(by=['method', 'req_count'])
sort_plan_df = df_filtered.sort_values(by=['method', 'plan_count'])

In [ ]:
fig = px.line(
    sort_plan_df,
    y='plan_count',
    x='req_count',
    # x='plan_count',
    # y='req_count',
    color='method',
    title='Request time vs. vehicle count',
    # text="req_count",
    facet_col='area_short',
    # facet_row='area_short',
)

area_order = list(fig.layout.annotations[i].text.split('=')[-1] for i in range(len(fig.layout.annotations)))

for i, area_name in enumerate(area_order, start=1):
    req_data = sort_df[sort_df["area_short"] == area_name]  # Filter data for the current category
    bumper = 10
    if area_name == 'SY':
        bumper = 500
    x_min = req_data["req_count"].min() - bumper
    x_max = req_data["req_count"].max() + bumper
    # x_min = req_data["plan_count"].min() - bumper
    # x_max = req_data["plan_count"].max() + bumper
    fig.update_xaxes(range=[x_min, x_max], col=i, matches=None, row=1)
    # fig.update_yaxes(range=[x_min, x_max], row=i, col=1, matches=None)
    

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.write_image(f"{IMG_PATH}/computational_time_vs_vehicle_count.png")
fig.show()